In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from difflib import SequenceMatcher

In [ ]:
# ---------------------------------------------------------
# 1. 유사도 측정 함수 정의 (Intensity 역추적용)
# ---------------------------------------------------------
def calculate_change_ratio(row):
    # 원문과 교정문 사이의 유사도(0.0 ~ 1.0) 계산
    # ratio()가 1.0이면 완전 동일, 0.0이면 완전 다름
    s = SequenceMatcher(None, str(row['original']), str(row['corrected']))
    return s.ratio()

def map_intensity(ratio):
    # 유사도에 따라 Intensity 매핑 (임의 기준 설정)
    if ratio > 0.9:       # 90% 이상 동일 -> 아주 조금 고침
        return "weak"
    elif ratio > 0.6:     # 60~90% 동일 -> 적당히 고침
        return "moderate"
    else:                 # 60% 미만 동일 -> 대거 수정함
        return "strong"

# ---------------------------------------------------------
# 2. 데이터 로드 및 전처리
# ---------------------------------------------------------
# 예시: 사용자가 보여준 head 데이터를 df_new라고 가정
# 실제로는 pd.read_json("문장교정기록.json") 등으로 로드
df_new = pd.read_csv('./data/munch_labeled_RESUMED_200125.csv')
df_new.dropna(subset=['difficulty_label'], inplace=True)   
# ---------------------------------------------------------
# 3. 결측 컬럼(Field, Intensity) 채우기
# ---------------------------------------------------------
print("⚙️ 메타데이터 자동 생성 중...")

# (1) Field: 정보가 없으므로 'none'로 통일 (안전한 선택)
df_new['field'] = 'none'

# (2) Intensity: 원문 vs 교정문 차이를 계산하여 자동 할당
# 유사도 계산
df_new['similarity'] = df_new.apply(calculate_change_ratio, axis=1)
# 강도 매핑
df_new['intensity'] = df_new['similarity'].apply(map_intensity)

# ---------------------------------------------------------
# 4. Feature Engineering (기존 모델 포맷 맞추기)
# ---------------------------------------------------------
# 컬럼명 통일 (original -> input_sentence)
df_new.rename(columns={'original': 'input_sentence'}, inplace=True)

# 기존 combine_features 함수 재사용
def combine_features(row):
    intensity_tag = f"[{str(row['intensity']).upper()}]"
    field_tag = f"[{str(row['field']).upper()}]"
    return f"{intensity_tag} {field_tag} {row['input_sentence']}"

df_new['text'] = df_new.apply(combine_features, axis=1)
df_new['label'] = df_new['difficulty_label'].astype(int)

# 결과 확인
print(df_new[['text', 'label', 'intensity', 'similarity']].head())

In [ ]:
print(df_new.head())

In [ ]:
df_new.to_csv('./data/munch_labeled_RESUMED_200125_revised.csv',index=False, encoding='utf-8-sig')